# Part A – Legal & Courtesy Amount Detection
**ICS472 – Natural Language Processing**  
**Team:** Mohammed Al Sheqaih · Abdulrhman Ammar

**Goal:** Train a YOLOv8 object detector to locate two regions in each check image:
- **Class 0 – Legal amount** (handwritten Arabic text region)
- **Class 1 – Courtesy amount** (digit region)

**Evaluation metrics:** Accuracy @ IoU ≥ {0.50, 0.75, 0.90} and Mean IoU

**Output:** Bounding-box predictions on the test set + saved crops for Parts B and C

## 1. Imports & Setup

In [1]:
import sys, os, shutil, json, random
sys.path.insert(0, os.path.abspath('.'))

import yaml
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import torch
from ultralytics import YOLO

from utils import (
    TRAIN_IMAGES, TEST_IMAGES, TRAIN_BBOX, TEST_BBOX, ARTIFACTS,
    parse_bbox, load_image, crop_region, yolo_to_xyxy, detection_metrics
)

# ── Device ──────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = 'cuda'
    print('CUDA is available. Using GPU.')
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'
print(f'Device: {DEVICE}')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

plt.rcParams['figure.dpi'] = 120

YOLO_DIR  = ARTIFACTS / 'yolo_dataset'
CROPS_DIR = ARTIFACTS / 'crops'
RUNS_DIR  = ARTIFACTS / 'yolo_runs'

CUDA is available. Using GPU.
Device: cuda


## 2. Prepare YOLO Dataset

YOLOv8 expects this layout:
```
yolo_dataset/
├── images/
│   ├── train/   (80 % of labelled images)
│   ├── val/     (20 % of labelled images)
│   └── test/
└── labels/
    ├── train/
    ├── val/
    └── test/
```
We use **symlinks** for images (avoids duplicating ~2 GB of TIFFs) and copy the small label `.txt` files.

`ac00048` is excluded — it has no bounding-box annotation.

In [2]:
MISSING = {'ac00048'}  # no bbox label

def build_yolo_dirs():
    # ── Wipe any stale dataset (e.g. old grayscale images) ───────────────
    if YOLO_DIR.exists():
        shutil.rmtree(YOLO_DIR)
    for split in ('train', 'val', 'test'):
        (YOLO_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
        (YOLO_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

def copy_as_rgb(src, dst):
    """Open any TIFF (including grayscale) and save as 3-channel RGB."""
    Image.open(src).convert('RGB').save(dst)

def populate_split(stems, split, img_src, lbl_src):
    for stem in tqdm(stems, desc=f'  {split}', leave=False):
        img_file = img_src / f'{stem}.tif'
        lbl_file = lbl_src / f'{stem}.txt'
        if img_file.exists():
            copy_as_rgb(img_file, YOLO_DIR / 'images' / split / f'{stem}.tif')
        if lbl_file.exists():
            shutil.copy2(lbl_file, YOLO_DIR / 'labels' / split / f'{stem}.txt')

# ── Collect labelled train stems ──────────────────────────────────────────
all_train_stems = sorted(
    p.stem for p in TRAIN_BBOX.glob('*.txt')
    if p.stem not in MISSING
)
random.shuffle(all_train_stems)
split_idx   = int(0.8 * len(all_train_stems))
train_stems = all_train_stems[:split_idx]
val_stems   = all_train_stems[split_idx:]
test_stems  = [p.stem for p in sorted(TEST_IMAGES.glob('*.tif'))]

print(f'Train: {len(train_stems)} | Val: {len(val_stems)} | Test: {len(test_stems)}')

# ── Build directory structure & copy as RGB ───────────────────────────────
build_yolo_dirs()
populate_split(train_stems, 'train', TRAIN_IMAGES, TRAIN_BBOX)
populate_split(val_stems,   'val',   TRAIN_IMAGES, TRAIN_BBOX)
populate_split(test_stems,  'test',  TEST_IMAGES,  TEST_BBOX)

print('Dataset directories ready (all images converted to RGB).')


Train: 1439 | Val: 360 | Test: 600


Dataset directories ready (all images converted to RGB).


In [3]:
# ── Write data.yaml ───────────────────────────────────────────────────────
data_yaml = {
    'path':  str(YOLO_DIR.resolve()),
    'train': 'images/train',
    'val':   'images/val',
    'test':  'images/test',
    'nc':    2,
    'names': {0: 'legal', 1: 'courtesy'},
}
yaml_path = YOLO_DIR / 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False, sort_keys=False)

print(f'data.yaml saved to: {yaml_path}')
print(open(yaml_path).read())

data.yaml saved to: C:\Users\Admin\Desktop\NLP_Project\arabic-bank-check-processing\codebase\artifacts\yolo_dataset\data.yaml
path: C:\Users\Admin\Desktop\NLP_Project\arabic-bank-check-processing\codebase\artifacts\yolo_dataset
train: images/train
val: images/val
test: images/test
nc: 2
names:
  0: legal
  1: courtesy



## 3. Train YOLOv8

We use **YOLOv8s** (small variant) pre-trained on COCO. Key choices:
- `imgsz=640` — good balance between speed and accuracy for this image size
- `epochs=100` — sufficient for fine-tuning; early stopping kicks in if val loss plateaus
- `batch=16` — adjust down to `8` if VRAM is tight
- `patience=20` — stops early if no improvement for 20 epochs

> **Runtime:** ~30–60 min on RTX 5060.

In [4]:
from tqdm.notebook import tqdm as tqdm_nb

model = YOLO('yolov8s.pt')  # downloads pretrained weights on first run

EPOCHS = 100
pbar  = tqdm_nb(total=EPOCHS, desc='Training', unit='epoch', dynamic_ncols=True)
_log  = []  # accumulate per-epoch metrics for the plot

def on_train_epoch_end(trainer):
    ep      = trainer.epoch + 1
    metrics = trainer.metrics
    loss    = trainer.loss.item() if hasattr(trainer.loss, 'item') else float(trainer.loss)
    map50   = metrics.get('metrics/mAP50(B)', float('nan'))
    map5095 = metrics.get('metrics/mAP50-95(B)', float('nan'))
    _log.append({'epoch': ep, 'loss': loss, 'mAP50': map50, 'mAP50-95': map5095})
    pbar.set_postfix(loss=f'{loss:.4f}', mAP50=f'{map50:.4f}', mAP50_95=f'{map5095:.4f}')
    pbar.update(1)
    tqdm_nb.write(
        f'Epoch {ep:>3}/{EPOCHS} | loss={loss:.4f} | mAP50={map50:.4f} | mAP50-95={map5095:.4f}'
    )

def on_train_end(trainer):
    pbar.close()

model.add_callback('on_train_epoch_end', on_train_epoch_end)
model.add_callback('on_train_end',       on_train_end)

train_results = model.train(
    data    = str(yaml_path),
    epochs  = EPOCHS,
    imgsz   = 640,
    batch   = 16,          # reduce to 8 if OOM
    device  = DEVICE,
    patience= 20,
    seed    = SEED,
    project = str(RUNS_DIR),
    name    = 'yolov8s_checks',
    exist_ok= True,
    verbose = False,
)

BEST_WEIGHTS = RUNS_DIR / 'yolov8s_checks' / 'weights' / 'best.pt'
print(f'\nBest weights: {BEST_WEIGHTS}')

# ── Training curve ────────────────────────────────────────────────────────
if _log:
    epochs_arr = [r['epoch']    for r in _log]
    losses     = [r['loss']     for r in _log]
    map50s     = [r['mAP50']    for r in _log]
    map5095s   = [r['mAP50-95'] for r in _log]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

    ax1.plot(epochs_arr, losses, color='steelblue')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax1.set_title('Training Loss'); ax1.grid(True, alpha=0.3)

    ax2.plot(epochs_arr, map50s,   label='mAP@0.50',     color='darkorange')
    ax2.plot(epochs_arr, map5095s, label='mAP@0.50:0.95', color='green')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('mAP')
    ax2.set_title('Validation mAP'); ax2.legend(); ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(ARTIFACTS / 'partA_training_curves.png', bbox_inches='tight')
    plt.show()


Training:   0%|          | 0/100 [00:00<?, ?epoch/s]

New https://pypi.org/project/ultralytics/8.4.48 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.14  Python-3.10.19 torch-2.12.0.dev20260219+cu128 CUDA:0 (NVIDIA GeForce RTX 5060, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\Admin\Desktop\NLP_Project\arabic-bank-check-processing\codebase\artifacts\yolo_dataset\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0

<Figure size 1560x480 with 2 Axes>

## 4. Validation Metrics (Built-in YOLO Report)

In [5]:
best_model = YOLO(str(BEST_WEIGHTS))
val_results = best_model.val(
    data   = str(yaml_path),
    split  = 'val',
    device = DEVICE,
    verbose= True,
)
print('mAP50  :', round(val_results.box.map50, 4))
print('mAP50-95:', round(val_results.box.map,  4))

Ultralytics 8.4.14  Python-3.10.19 torch-2.12.0.dev20260219+cu128 CUDA:0 (NVIDIA GeForce RTX 5060, 8151MiB)
Model summary (fused): 73 layers, 11,126,358 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 228.783.8 MB/s, size: 242.5 KB)
val: Scanning C:\Users\Admin\Desktop\NLP_Project\arabic-bank-check-processing\codebase\artifacts\yolo_dataset\labels\val.cache... 360 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 360/360  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 9.2it/s 2.5s<0.2s
                   all        360        720      0.896      0.919      0.934      0.607
                 legal        360        360      0.951      0.981      0.975      0.645
              courtesy        360        360      0.841      0.858      0.894      0.569
Speed: 0.5ms preprocess, 2.6ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to C:\Users\Admin\Desktop\NLP_Proje

## 5. Custom Evaluation on Test Set

We compute the project-specific metrics on the **test set**:
- Accuracy @ IoU ≥ 0.50 / 0.75 / 0.90
- Mean IoU

Both classes (legal + courtesy) are evaluated together as the assignment specifies.

In [9]:
test_imgs = sorted(TEST_IMAGES.glob('*.tif'))

predictions    = []
ground_truths  = []
pred_boxes_raw = []  # store raw xyxy predictions for crop saving

for img_path in tqdm(test_imgs, desc='Evaluating test set'):
    img      = Image.open(img_path).convert('RGB')  # ensure 3-channel for YOLO
    W, H     = img.size

    # ── Model prediction ──────────────────────────────────────────────
    result   = best_model(img, verbose=False)[0]   # pass PIL image, not path
    pred     = {'courtesy': None, 'legal': None}
    pred_raw = {}
    for box in result.boxes:
        cls_id = int(box.cls)
        xyxy   = [round(v, 1) for v in box.xyxy[0].tolist()]
        key    = 'courtesy' if cls_id == 1 else 'legal'
        # keep the highest-confidence box per class
        if pred[key] is None:
            pred[key]     = xyxy
            pred_raw[key] = xyxy
    predictions.append(pred)
    pred_boxes_raw.append({'stem': img_path.stem, 'boxes': pred_raw, 'size': (W, H)})

    # ── Ground truth ──────────────────────────────────────────────────
    gt      = {'courtesy': None, 'legal': None}
    gt_path = TEST_BBOX / (img_path.stem + '.txt')
    if gt_path.exists():
        for cls, cx, cy, bw, bh in parse_bbox(gt_path):
            key     = 'courtesy' if cls == 1 else 'legal'
            gt[key] = yolo_to_xyxy(cx, cy, bw, bh, W, H)
    ground_truths.append(gt)

# ── Compute metrics ───────────────────────────────────────────────────
metrics = detection_metrics(predictions, ground_truths, thresholds=(0.5, 0.75, 0.9))
print('\n── Test Set Detection Metrics ──')
for k, v in metrics.items():
    print(f'  {k}: {v}')


Evaluating test set: 100%|██████████| 600/600 [00:10<00:00, 58.01it/s]


── Test Set Detection Metrics ──
  acc@0.5: 67.25
  acc@0.75: 20.92
  acc@0.9: 2.75
  mean_iou: 58.2


In [10]:
# ── Per-class IoU breakdown ────────────────────────────────────────────────
ca_ious, la_ious = [], []

from utils import iou
for pred, gt in zip(predictions, ground_truths):
    for key, lst in (('courtesy', ca_ious), ('legal', la_ious)):
        if pred[key] and gt[key]:
            lst.append(iou(pred[key], gt[key]))
        else:
            lst.append(0.0)

print(f'Courtesy — Mean IoU: {np.mean(ca_ious)*100:.2f}%')
print(f'Legal    — Mean IoU: {np.mean(la_ious)*100:.2f}%')

# ── Save metrics to disk ──────────────────────────────────────────────────
metrics_out = {
    **metrics,
    'courtesy_mean_iou': round(float(np.mean(ca_ious)) * 100, 2),
    'legal_mean_iou':    round(float(np.mean(la_ious)) * 100, 2),
}
with open(ARTIFACTS / 'partA_metrics.json', 'w') as f:
    json.dump(metrics_out, f, indent=2)
print('\nMetrics saved to artifacts/partA_metrics.json')

Courtesy — Mean IoU: 46.02%
Legal    — Mean IoU: 70.37%

Metrics saved to artifacts/partA_metrics.json


## 6. Visualise Predictions vs Ground Truth

In [11]:
def draw_boxes(ax, img, pred, gt, title=''):
    ax.imshow(img, cmap='gray')
    styles = [
        ('pred',  pred, 'lime',  '-'),
        ('gt',    gt,   'red',   '--'),
    ]
    for label_prefix, boxes, color, ls in styles:
        for key, box in boxes.items():
            if box is None:
                continue
            x1, y1, x2, y2 = box
            rect = patches.Rectangle(
                (x1, y1), x2-x1, y2-y1,
                linewidth=2, edgecolor=color, facecolor='none', linestyle=ls
            )
            ax.add_patch(rect)
            ax.text(x1, y1 - 4, f'{label_prefix}:{key}', color=color, fontsize=7)
    ax.axis('off')
    ax.set_title(title, fontsize=9)

fig, axes = plt.subplots(4, 1, figsize=(13, 16))
sample_idxs = [0, 5, 20, 50]
for ax, idx in zip(axes, sample_idxs):
    img = Image.open(test_imgs[idx])
    draw_boxes(ax, img, predictions[idx], ground_truths[idx],
               title=test_imgs[idx].stem)
plt.suptitle('Predictions (green) vs Ground Truth (red)', y=1.01, fontsize=11)
plt.tight_layout()
plt.savefig(ARTIFACTS / 'partA_predictions.png', bbox_inches='tight')
plt.show()

<Figure size 1560x1920 with 4 Axes>

## 7. Save Crops for Parts B and C

- **Train crops** — cropped using **ground-truth** bboxes (clean signal for training the recognisers)
- **Test crops**  — cropped using **model predictions** (simulates the real end-to-end pipeline)

Files are named `Cac#####.tif` (courtesy) and `Lac#####.tif` (legal) to match the annotation files.

In [12]:
for split in ('train', 'test'):
    for region in ('courtesy', 'legal'):
        (CROPS_DIR / split / region).mkdir(parents=True, exist_ok=True)

def save_crop(img_path, cx, cy, bw, bh, out_path):
    img  = load_image(img_path)
    crop = crop_region(img, cx, cy, bw, bh)
    crop.save(out_path)

# ── Train crops (ground truth bboxes) ─────────────────────────────────────
train_img_paths = sorted(TRAIN_IMAGES.glob('*.tif'))
skipped = 0
for img_path in tqdm(train_img_paths, desc='Saving train crops'):
    if img_path.stem in MISSING:
        skipped += 1
        continue
    lbl_path = TRAIN_BBOX / (img_path.stem + '.txt')
    if not lbl_path.exists():
        skipped += 1
        continue
    for cls, cx, cy, bw, bh in parse_bbox(lbl_path):
        region  = 'courtesy' if cls == 1 else 'legal'
        prefix  = 'C' if cls == 1 else 'L'
        out     = CROPS_DIR / 'train' / region / f'{prefix}{img_path.stem}.tif'
        save_crop(img_path, cx, cy, bw, bh, out)
print(f'Train crops done. Skipped: {skipped}')

Saving train crops: 100%|██████████| 1800/1800 [00:30<00:00, 59.26it/s]

Train crops done. Skipped: 1


In [13]:
# ── Test crops (model predictions) ────────────────────────────────────────
missing_preds = 0
for entry in tqdm(pred_boxes_raw, desc='Saving test crops'):
    stem     = entry['stem']
    img_path = TEST_IMAGES / f'{stem}.tif'
    img      = load_image(img_path)
    W, H     = img.size

    for key, (cls_id, prefix) in {
        'courtesy': (1, 'C'),
        'legal':    (0, 'L'),
    }.items():
        box = entry['boxes'].get(key)
        if box is None:
            # fall back to ground truth if model missed the box
            gt_path = TEST_BBOX / f'{stem}.txt'
            if gt_path.exists():
                for c, cx, cy, bw, bh in parse_bbox(gt_path):
                    if c == cls_id:
                        box = yolo_to_xyxy(cx, cy, bw, bh, W, H)
                        break
            if box is None:
                missing_preds += 1
                continue
        x1, y1, x2, y2 = [int(v) for v in box]
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(W, x2), min(H, y2)
        crop    = img.crop((x1, y1, x2, y2))
        out     = CROPS_DIR / 'test' / key / f'{prefix}{stem}.tif'
        crop.save(out)

print(f'Test crops done. Missing predictions (fell back to GT): {missing_preds}')

Saving test crops: 100%|██████████| 600/600 [00:02<00:00, 239.35it/s]

Test crops done. Missing predictions (fell back to GT): 0


## 8. Summary

In [14]:
train_ca_crops = list((CROPS_DIR / 'train' / 'courtesy').glob('*.tif'))
train_la_crops = list((CROPS_DIR / 'train' / 'legal').glob('*.tif'))
test_ca_crops  = list((CROPS_DIR / 'test'  / 'courtesy').glob('*.tif'))
test_la_crops  = list((CROPS_DIR / 'test'  / 'legal').glob('*.tif'))

print('── Part A Complete ──')
print(f"  Acc @ IoU≥0.50 : {metrics['acc@0.5']}%")
print(f"  Acc @ IoU≥0.75 : {metrics['acc@0.75']}%")
print(f"  Acc @ IoU≥0.90 : {metrics['acc@0.9']}%")
print(f"  Mean IoU       : {metrics['mean_iou']}%")
print()
print(f'  Crops saved:')
print(f'    Train courtesy : {len(train_ca_crops)}')
print(f'    Train legal    : {len(train_la_crops)}')
print(f'    Test  courtesy : {len(test_ca_crops)}')
print(f'    Test  legal    : {len(test_la_crops)}')
print()
print('  → Run 02_Part_B_Courtesy.ipynb next')

── Part A Complete ──
  Acc @ IoU≥0.50 : 67.25%
  Acc @ IoU≥0.75 : 20.92%
  Acc @ IoU≥0.90 : 2.75%
  Mean IoU       : 58.2%

  Crops saved:
    Train courtesy : 1799
    Train legal    : 1799
    Test  courtesy : 600
    Test  legal    : 600

  → Run 02_Part_B_Courtesy.ipynb next
